In [ ]:
# 导入需要的库
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re
from collections import Counter
import networkx as nx
from datetime import datetime
import warnings
from wordcloud import WordCloud
from statsmodels.tsa.seasonal import seasonal_decompose
from textblob import TextBlob
warnings.filterwarnings('ignore')

# 设置图表样式
plt.style.use('ggplot')
sns.set(style="whitegrid")

nltk.download('vader_lexicon')
nltk.download('punkt')
nltk.download('stopwords')


# 数据预处理

In [ ]:
# 加载三个数据集
businesses = pd.read_csv("D:\\FDSM\\2025Spring\\程序设计与可视化\\期末作业\\business_df.csv")
reviews = pd.read_csv("D:\\FDSM\\2025Spring\\程序设计与可视化\\期末作业\\review_df.csv")
users = pd.read_csv("D:\\FDSM\\2025Spring\\程序设计与可视化\\期末作业\\user_df.csv")

print(f"用户数据: {users.shape[0]} 行")
print(f"评论数据: {reviews.shape[0]} 行")
print(f"商家数据: {businesses.shape[0]} 行")

# 检查数据基本情况
print(users.head())
print(reviews.head())
print(businesses.head())

print(businesses.dtypes)
print(reviews.dtypes)
print(users.dtypes)

# 数据清洗


In [ ]:
# 将评论日期转换为日期格式
reviews['date'] = pd.to_datetime(reviews['date'])
# 将用户注册日期转换为日期格式
users['yelping_since'] = pd.to_datetime(users['yelping_since'])
# 检查缺失值
print('检查缺失值')
print(reviews.isna().mean(),'\n')
print(users.isna().mean(),'\n')
print(businesses.isna().mean(),'\n')

# 将商家根据Categories列确定主要类别

In [ ]:
businesses['categories'].head()

In [ ]:

category_mapping = {
    'Restaurants': 'Food & Dining',
    'Food': 'Food & Dining', 
    'Coffee & Tea': 'Food & Dining',
    'Cafes': 'Food & Dining',
    'Fast Food': 'Food & Dining',
    'Bakeries': 'Food & Dining',
    'Bars': 'Food & Dining',
    'Pizza': 'Food & Dining',
    'Breakfast & Brunch': 'Food & Dining',
    'Burgers': 'Food & Dining',
    'Sandwiches': 'Food & Dining',
    'Seafood': 'Food & Dining',
    'Steakhouses': 'Food & Dining',
    'Sushi Bars': 'Food & Dining',
    'Juice Bars & Smoothies': 'Food & Dining',
    'Desserts': 'Food & Dining',
    'Ice Cream & Frozen Yogurt': 'Food & Dining',
    
    'Shopping': 'Retail',
    'Grocery': 'Retail',
    'Convenience Stores': 'Retail',
    'Fashion': 'Retail',
    'Clothing': 'Retail',
    'Department Stores': 'Retail',
    'Electronics': 'Retail',
    'Furniture Stores': 'Retail',
    'Home & Garden': 'Retail',
    'Hardware Stores': 'Retail',
    
    'Beauty & Spas': 'Personal Services',
    'Hair Salons': 'Personal Services',
    'Nail Salons': 'Personal Services',
    'Day Spas': 'Personal Services',
    'Massage': 'Personal Services',
    'Barbers': 'Personal Services',
    
    'Health & Medical': 'Healthcare',
    'Doctors': 'Healthcare',
    'Dentists': 'Healthcare',
    'Medical Centers': 'Healthcare',
    'Optometrists': 'Healthcare',
    'Hospitals': 'Healthcare',
    'Urgent Care': 'Healthcare',
    
    'Home Services': 'Home Services',
    'Contractors': 'Home Services',
    'Plumbing': 'Home Services',
    'Electricians': 'Home Services',
    'Landscaping': 'Home Services',
    'Cleaning': 'Home Services',
    'HVAC': 'Home Services',
    'Movers': 'Home Services',
    
    'Automotive': 'Automotive',
    'Auto Repair': 'Automotive',
    'Car Dealers': 'Automotive',
    'Oil Change Stations': 'Automotive',
    'Car Wash': 'Automotive',
    'Gas Stations': 'Automotive',
    
    'Hotels & Travel': 'Travel & Hospitality',
    'Hotels': 'Travel & Hospitality',
    'Airports': 'Travel & Hospitality',
    'Travel Agencies': 'Travel & Hospitality',
    'Vacation Rentals': 'Travel & Hospitality',
    
    'Local Services': 'Local Services',
    'Dry Cleaning': 'Local Services',
    'Pet Services': 'Local Services',
    'Pet Groomers': 'Local Services',
    'Dog Walkers': 'Local Services'
}

# 提取categories的第一个词
businesses['category_first_word'] = businesses['categories'].apply(lambda x: x.split(' ')[0])

# 为每个商家分配大类别组
def assign_category_group(main_category, category_first_word):
    # 检查categories中是否包含映射表中的关键词
    for key, value in category_mapping.items():
        if key.lower() in main_category.lower():
            return value
    
    # 默认分类为"Other"
    return 'Other'

# 应用分类函数
businesses['category_group'] = businesses.apply(
    lambda row: assign_category_group(row['categories'], row['category_first_word']), axis=1
)

# 检查大类别组分布
category_group_counts = businesses['category_group'].value_counts()
print("\n商家大类别分布:")
print(category_group_counts)

# 选择样本量较大的5个类别组
main_category_groups = category_group_counts.head(5).index.tolist()
print(f"\n主要商家类别组: {main_category_groups}")

# 数据合并

In [ ]:
# 数据合并
# 先合并评论和用户数据
reviews_users = pd.merge(reviews, users, on='user_id', suffixes=('', '_user'))
# 再合并商家数据
full_data = pd.merge(reviews_users, businesses, on='business_id', suffixes=('', '_business'))

# 探索性分析

In [ ]:
# 选择商家评论总数最多的五个城市为主要城市
top_cities = businesses.groupby('city')['review_count'].sum().sort_values(ascending=False).head(5).index.tolist()
print(f"\n分析的主要城市: {top_cities}")

city_performance = businesses[businesses['city'].isin(top_cities)].groupby('city').agg({
    'stars': 'mean',
    'review_count': 'mean'
}).reset_index()

category_performance = businesses[businesses['category_group'].isin(main_category_groups)].groupby('category_group').agg({
    'stars': 'mean',
    'review_count': 'mean'
}).reset_index()

# 创建一个 2x2 的子图网格
fig, axes = plt.subplots(2, 2, figsize=(16, 12))  

# 子图1：各城市商家平均评分
sns.barplot(x='city', y='stars', data=city_performance, palette='viridis', ax=axes[0, 0])
axes[0, 0].set_title('Avg. Stars for Cities', fontsize=14)
axes[0, 0].set_xlabel('City', fontsize=12)
axes[0, 0].set_ylabel('Avg. Stars', fontsize=12)
axes[0, 0].tick_params(axis='x', rotation=45)

# 子图2：各城市商家平均评论数
sns.barplot(x='city', y='review_count', data=city_performance, palette='magma', ax=axes[0, 1])
axes[0, 1].set_title('Avg. # of Comments', fontsize=14)
axes[0, 1].set_xlabel('City', fontsize=12)
axes[0, 1].set_ylabel('# of Comments', fontsize=12)
axes[0, 1].tick_params(axis='x', rotation=45)

# 子图3：各类别商家平均评分
sns.barplot(x='category_group', y='stars', data=category_performance, palette='viridis', ax=axes[1, 0])
axes[1, 0].set_title('Stars by Category', fontsize=14)
axes[1, 0].set_xlabel('Category', fontsize=12)
axes[1, 0].set_ylabel('Avg. Stars', fontsize=12)
axes[1, 0].tick_params(axis='x', rotation=45)

# 子图4：各类别商家平均评论数
sns.barplot(x='category_group', y='review_count', data=category_performance, palette='magma', ax=axes[1, 1])
axes[1, 1].set_title('# of Comments by Category', fontsize=14)
axes[1, 1].set_xlabel('Category', fontsize=12)
axes[1, 1].set_ylabel('# of Comments', fontsize=12)
axes[1, 1].tick_params(axis='x', rotation=45)

# 自动调整布局
plt.tight_layout()
plt.show()


In [ ]:
main_businesses = businesses[businesses['category_group'].isin(main_category_groups)]
#对主要类别的商家的评分可视化
plt.figure(figsize=(10, 6))
sns.boxplot(x='category_group', y='stars', data=main_businesses)
plt.title('Boxplot of Ratings by Categories', fontsize=16)
plt.xlabel('category_group', fontsize=12)
plt.ylabel('Stars Rating', fontsize=12)
plt.tight_layout()
plt.show()

# 文本分析

## 文本数据探索性分析

In [ ]:
# 查看评分分布
plt.figure(figsize=(10, 6))
sns.countplot(x='stars', data=reviews, palette='viridis')
plt.title('Distribution of Review Star Ratings')
plt.xlabel('Star Rating')
plt.ylabel('Count')
plt.show()

# 检查评论长度分布
reviews['text_length'] = reviews['text'].apply(len)
plt.figure(figsize=(12, 6))
sns.histplot(reviews['text_length'], bins=50, kde=True)
plt.title('Distribution of Review Text Length')
plt.xlabel('Text Length (characters)')
plt.ylabel('Count')
plt.xlim(0, 2000) 
plt.show()
skewness = reviews['text_length'].skew()
kurtosis = reviews['text_length'].kurt()
print(f"偏度（Skewness）: {skewness}")
print(f"峰度（Kurtosis）: {kurtosis}")


# 检查不同城市的评论数量
city_review_counts = full_data.groupby('city').size().reset_index(name='review_count')
top_cities = city_review_counts.sort_values('review_count', ascending=False).head(10)

plt.figure(figsize=(12, 6))
sns.barplot(x='review_count', y='city', data=top_cities, palette='viridis')
plt.title('Top 10 Cities by Number of Reviews')
plt.xlabel('Number of Reviews')
plt.ylabel('City')
plt.tight_layout()
plt.show()

## 文本预处理

In [ ]:
def preprocess_text(text):
    # 转换为小写
    text = text.lower()
    # 去除特殊字符和数字
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # 分词
    tokens = word_tokenize(text)
    # 停用词过滤
    stop_words = set(stopwords.words('english'))
    custom_stop_words = {'restaurant', 'place', 'food', 'really', 'good', 'great', 'just', 'like', 'get', 'one', 'also', 'im', 'us', 'even', 'would', 'could', 'cant'}
    stop_words.update(custom_stop_words)
    tokens = [token for token in tokens if token not in stop_words and len(token) > 2]
    # 词形还原
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(token) for token in tokens]
    return ' '.join(tokens)

#总表和分表都要处理
full_data['processed_text'] = full_data['text'].apply(preprocess_text)
reviews['processed_text'] = reviews['text'].apply(preprocess_text)

## TF-IDF词频分析

In [ ]:
#提取文字
texts = full_data['text'].fillna('').tolist()

#建立向量化vectorizer
vectorizer = TfidfVectorizer(stop_words='english', max_features=1000)  # 可调大小
X = vectorizer.fit_transform(texts)

#提取
feature_names = vectorizer.get_feature_names_out()
tfidf_means = X.mean(axis=0).A1  # 变成1维数组
tfidf_df = pd.DataFrame({'term': feature_names, 'score': tfidf_means})
top_tfidf = tfidf_df.sort_values(by='score', ascending=False).head(30)    #选频次最高的可视化

#词云
wc = WordCloud(width=800, height=400, background_color='white')
wc.generate_from_frequencies(dict(zip(top_tfidf['term'], top_tfidf['score'])))

#可视化
plt.figure(figsize=(12, 6))
plt.imshow(wc, interpolation='bilinear')
plt.axis('off')
plt.title("Top TF-IDF Keywords from Yelp Reviews")
plt.show()

plt.figure(figsize=(10, 6))
sns.barplot(data=top_tfidf, x='score', y='term', palette='viridis')
plt.title("Top TF-IDF Terms in Yelp Reviews")
plt.xlabel("TF-IDF Score")
plt.ylabel("Term")
plt.tight_layout()
plt.show()


## 城市热词

In [ ]:
top_cities = full_data.groupby('city')['review_count'].sum().sort_values(ascending=False).head(10).index.tolist()

In [ ]:
from geopy.geocoders import Nominatim
import time
#调取城市坐标
geolocator = Nominatim(user_agent="yelp_map")
city_coords = {}
for city in top_cities:
    try:
        location = geolocator.geocode(f"{city}, USA")
        if location:
            city_coords[city] = (location.longitude, location.latitude)
        else:
            print(f"未找到坐标: {city}")
        time.sleep(1)  
    except Exception as e:
        print(f"错误: {city} -> {e}")

In [ ]:
from collections import Counter
city_labels = {}
#按城市提取高频词
for city in top_cities:
    texts = full_data[full_data['city'] == city]['processed_text'].dropna().tolist()
    all_words = ' '.join(texts).split()
    top_words = Counter(all_words).most_common(5)
    word_str = ', '.join([w for w, _ in top_words])
    city_labels[city] = word_str


In [ ]:
fig = plt.figure(figsize=(18, 12))
ax = plt.axes(projection=ccrs.LambertConformal())
ax.set_extent([-125, -65, 24, 50], crs=ccrs.Geodetic())

# 添加地图背景
ax.stock_img()
ax.add_feature(cfeature.STATES.with_scale('50m'), edgecolor='gray')
ax.add_feature(cfeature.BORDERS, linestyle='-', edgecolor='black')
ax.add_feature(cfeature.LAKES, color='lightblue', alpha=0.5)
ax.add_feature(cfeature.LAND, color='#f0eada')

# 标记城市与词汇
for city, (lon, lat) in city_coords.items():
    ax.plot(lon, lat, marker='o', color='crimson', markersize=7, transform=ccrs.Geodetic())
    ax.text(
        lon + 1, lat + 0.7,
        f"{city}\n{city_labels.get(city, '')}",
        transform=ccrs.Geodetic(),
        fontsize=17,
        color='darkblue',
        weight='bold',
        bbox=dict(boxstyle="round,pad=0.3", facecolor='white', alpha=0.85)
    )

plt.title("Top 10 Cities and Frequent Yelp Words", fontsize=20, weight='bold')
plt.tight_layout()
plt.show()


## 情感分析

In [ ]:
#情感打分器
sid = SentimentIntensityAnalyzer()

# 准备数据
review_sample = full_data

# 添加情感判别结果列
review_sample['sentiment_scores'] = review_sample['text'].apply(lambda x: sid.polarity_scores(str(x)))
review_sample['sentiment_compound'] = review_sample['sentiment_scores'].apply(lambda x: x['compound'])
review_sample['sentiment'] = review_sample['sentiment_compound'].apply(
    lambda x: 'positive' if x >= 0.05 else ('negative' if x <= -0.05 else 'neutral'))

# 分析不同类别商家的情感分布
category_sentiment = review_sample[review_sample['category_group'].isin(main_category_groups)].groupby(['category_group', 'sentiment']).size().unstack().fillna(0)
category_sentiment_pct = category_sentiment.div(category_sentiment.sum(axis=1), axis=0) * 100

print(category_sentiment_pct)

# 分析不同类别商家的情感
plt.figure(figsize=(12, 8))
category_sentiment_pct.plot(kind='bar', stacked=True, colormap='viridis')
plt.title('Sentiment Distribution Across Main Categories', fontsize=15)
plt.xlabel('Business Category', fontsize=12)
plt.ylabel('Percentage (%)', fontsize=12)
plt.legend(title='Sentiment')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# 不同评分的评论的情感打分分布
plt.figure(figsize=(12, 6))
sns.boxplot(data=review_sample, x='stars', y='sentiment_compound', palette='coolwarm')
plt.title('Sentiment Distribution Across Different Ratings', fontsize=16)
plt.xlabel('Ratings (Stars)', fontsize=12)
plt.ylabel('Sentiment Score (Compound)', fontsize=12)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# 不同城市的情感打分对比
filtered_data = full_data[full_data['city'].isin(top_cities)]
# 计算平均情绪得分
city_avg_sentiment = (
    filtered_data.groupby('city')['sentiment_compound']
    .mean()
    .sort_values(ascending=False)
)
# 绘图
plt.figure(figsize=(12, 6))
sns.barplot(x=city_avg_sentiment.values, y=city_avg_sentiment.index, palette='coolwarm')
plt.title('Comparison of Average Sentiment Scores for Selected Cities', fontsize=16)
plt.xlabel('Average Sentiment Score (Compound)')
plt.ylabel('City')
plt.tight_layout()
plt.show()


## 主题模型分析

In [ ]:
# 创建文档-词项矩阵，用countvectorizer而非tfidf
vectorizer = CountVectorizer(max_features=1000, max_df=0.95, min_df=2)
dtm = vectorizer.fit_transform(review_sample['processed_text'])
feature_names = vectorizer.get_feature_names_out()

# 应用LDA主题模型，5个主题
n_topics = 5
lda = LatentDirichletAllocation(n_components=n_topics, random_state=42, max_iter=10)
lda.fit(dtm)

# 打印主题词
for i, topic in enumerate(lda.components_):
    print(f"主题 #{i}:")
    topic_words = [feature_names[idx] for idx in topic.argsort()[-10:]]   ##最后十个主题词
    print(", ".join(topic_words))
    print()
doc_topic = lda.transform(dtm)
review_sample['dominant_topic'] = doc_topic.argmax(axis=1)

# 可视化主题分布
plt.figure(figsize=(10, 6))
sns.countplot(x='dominant_topic', data=review_sample, palette='Spectral')
plt.title('Review Topic Distribution', fontsize=15)
plt.xlabel('Topic Number', fontsize=12)
plt.ylabel('Number of Reviews', fontsize=12)
plt.tight_layout()
plt.show()

# 不同类别商家的主题偏好
category_topics = review_sample[review_sample['category_group'].isin(main_category_groups)].groupby(['category_group', 'dominant_topic']).size().unstack().fillna(0)
category_topics_pct = category_topics.div(category_topics.sum(axis=1), axis=0) * 100
plt.figure(figsize=(14, 8))
category_topics_pct.plot(kind='bar', stacked=True, colormap='tab10')
plt.title('Category Topic Preferences', fontsize=15)
plt.xlabel('Category', fontsize=12)
plt.ylabel('Percentage (%)', fontsize=12)
plt.legend(title='Topic Number')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# 网络效应分析

In [ ]:
# 在Yelp数据中，friends字段是以逗号分隔的用户ID字符串
def extract_friends(friends_str):
    if isinstance(friends_str, str):
        if friends_str == 'None' or friends_str == '':
            return []
        return [friend.strip() for friend in friends_str.split(',')]
    return []

# 将朋友关系转换为列表格式
full_data['friends_list'] = full_data['friends'].apply(extract_friends)

# 计算每个用户的朋友数量
full_data['friends_count'] = full_data['friends_list'].apply(len)

# 分析朋友圈评分一致性
user_avg_ratings = full_data.groupby('user_id')['stars'].mean().to_dict()
# 对于每条评论，计算用户评分与朋友平均评分的相关性
user_friend_ratings = []
for _, row in full_data.iterrows():
    user_id = row['user_id']
    user_rating = row['stars']
    friends = extract_friends(row['friends'])
    # 计算朋友的平均评分
    friend_ratings = [user_avg_ratings.get(friend, 0) for friend in friends if friend in user_avg_ratings]
    if friend_ratings:
        avg_friend_rating = sum(friend_ratings) / len(friend_ratings)
    else:
        avg_friend_rating = None
    user_friend_ratings.append({
        'user_id': user_id,
        'user_rating': user_rating,
        'friend_avg_rating': avg_friend_rating,
        'friends_count': len(friends)
    })

# 创建DataFrame并计算相关性
friend_rating_df = pd.DataFrame(user_friend_ratings)
correlation = friend_rating_df['user_rating'].corr(friend_rating_df['friend_avg_rating'])
plt.figure(figsize=(10, 8))
sns.regplot(x='friend_avg_rating', y='user_rating', data=friend_rating_df, scatter_kws={'alpha':0.3})
plt.title(f'Correlation between Friend Ratings and User Ratings\nCorrelation: {correlation:.4f}', fontsize=16)
plt.xlabel('Friend Scores', fontsize=12)
plt.ylabel('User Score', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 分析朋友关系网络密度
import networkx as nx
# 取评论数量最多的前1000个用户
top_users = full_data['user_id'].value_counts().head(1000).index.tolist()
filtered_data = full_data[full_data['user_id'].isin(top_users)]
# 创建社交网络图
G = nx.Graph()
# 添加节点
for user_id in top_users:
    user_name = users.loc[users['user_id'] == user_id, 'name'].values[0]
    G.add_node(user_id, name=user_name)

# 添加边（朋友关系）
for _, row in filtered_data[['user_id', 'friends']].drop_duplicates('user_id').iterrows():
    user = row['user_id']
    friends = extract_friends(row['friends'])
    for friend in friends:
        if friend in top_users:  
            G.add_edge(user, friend)

# 计算网络指标
density = nx.density(G)
avg_clustering = nx.average_clustering(G)

# 找出最大连通分量
largest_cc = max(nx.connected_components(G), key=len)
largest_cc_subgraph = G.subgraph(largest_cc)

# 计算中心性指标
degree_centrality = nx.degree_centrality(largest_cc_subgraph)
betweenness_centrality = nx.betweenness_centrality(largest_cc_subgraph)

# 输出网络分析结果
print(f"网络密度: {density:.4f}")
print(f"平均聚类系数: {avg_clustering:.4f}")
print(f"最大连通分量节点数: {len(largest_cc)}")
print(f"最大连通分量边数: {largest_cc_subgraph.number_of_edges()}")

print("\n度中心性前10的用户:")
for user, centrality in sorted(degree_centrality.items(), key=lambda x: x[1], reverse=True)[:10]:
    user_name = users.loc[users['user_id'] == user, 'name'].values[0]
    print(f"用户ID: {user}, 用户名: {user_name}, 度中心性: {centrality:.4f}")
    
print("\n中介中心性前10的用户:")
for user, centrality in sorted(betweenness_centrality.items(), key=lambda x: x[1], reverse=True)[:10]:
    user_name = users.loc[users['user_id'] == user, 'name'].values[0]
    print(f"用户ID: {user}, 用户名: {user_name}, 中介中心性: {centrality:.4f}")

# 绘制社交网络图
plt.figure(figsize=(15, 15))

# 选择前50个度中心性最高的节点
top_degree_nodes = sorted(degree_centrality.items(), key=lambda x: x[1], reverse=True)[:50]
top_degree_nodes = [node[0] for node in top_degree_nodes]

# 创建子图
subgraph = largest_cc_subgraph.subgraph(top_degree_nodes)

# 设置节点大小基于度中心性
node_size = [degree_centrality[node] * 5000 for node in subgraph.nodes()]

# 使用spring布局
pos = nx.spring_layout(subgraph, seed=42)

# 绘制网络图
nx.draw_networkx_nodes(subgraph, pos, node_size=node_size, node_color='lightblue', alpha=0.8)
nx.draw_networkx_edges(subgraph, pos, width=0.5, alpha=0.5)
nx.draw_networkx_labels(subgraph, pos, labels=nx.get_node_attributes(subgraph, 'name'), font_size=8)

plt.title('Yelp User Network', fontsize=16)
plt.axis('off')
plt.tight_layout()
plt.show()

# 是否为度中心性前10的对比

In [ ]:
# 创建度中心性Top10用户集合
top_10_degree_users = set([user for user, _ in sorted(degree_centrality.items(), key=lambda x: x[1], reverse=True)[:10]])

# 标记用户是否为Top10度中心性
full_data['is_top'] = full_data['user_id'].apply(lambda x: x in top_10_degree_users)

# 画小提琴图比较评分差异
plt.figure(figsize=(8, 6))
sns.violinplot(x='is_top', y='stars', data=full_data, inner='box', cut=0)
plt.xlabel('Is Top 10 Degree Centrality User')
plt.ylabel('Star Rating')
plt.title('Comparison of Ratings Between Top 10 Central Users and Others')
plt.xticks([0, 1], ['Others', 'Top 10 Users'])
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


# 商家成功率预测

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import re
from sklearn.feature_selection import RFE
from sklearn.decomposition import PCA
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# 计算 success_score
businesses['success_score'] = businesses['stars'] * np.log1p(businesses['review_count'])

# 计算 weighted_stars
businesses['weighted_stars'] = (businesses['stars'] * businesses['review_count']) / (businesses['review_count'] + 1)

# 查看新指标的分布
sns.histplot(businesses['success_score'], kde=True)
plt.title('Distribution of Success Score')  # 修改标题为英文
plt.xlabel('Success Score')  # 添加x轴标签
plt.ylabel('Frequency')  # 添加y轴标签
plt.tight_layout()
plt.show()

sns.histplot(businesses['weighted_stars'], kde=True)
plt.title('Distribution of Weighted Stars')  # 修改标题为英文
plt.xlabel('Weighted Stars')  # 添加x轴标签
plt.ylabel('Frequency')  # 添加y轴标签

plt.tight_layout()
plt.show()

# 查看两个指标的相关性
correlation = businesses['success_score'].corr(businesses['weighted_stars'])
print("Correlation between Success Score and Weighted Stars:", correlation)


# 特征工程

In [ ]:
# 合并数据
review_stats = reviews.groupby('business_id').agg({
    'stars': ['mean', 'std', 'count'],
    'date': ['min', 'max']
}).reset_index()

# 设置列名
review_stats.columns = ['business_id', 'avg_review_stars', 'std_review_stars', 
                        'total_reviews', 'first_review_date', 'last_review_date']

# 转换为日期类型
review_stats['first_review_date'] = pd.to_datetime(review_stats['first_review_date'])
review_stats['last_review_date'] = pd.to_datetime(review_stats['last_review_date'])

# 计算业务存在时间
review_stats['business_age_days'] = (review_stats['last_review_date'] - 
                                     review_stats['first_review_date']).dt.days
# 计算每日平均评论数
review_stats['reviews_per_day'] = review_stats['total_reviews'] / np.maximum(review_stats['business_age_days'], 1)

# 创建地理特征
businesses['is_in_major_city'] = businesses['city'].isin(top_cities)
business_features = pd.merge(businesses, review_stats, on='business_id', how='left')

print("特征数据形状:", business_features.shape)
business_features.head()

In [ ]:
# 查看目标变量与主要特征的关系
plt.figure(figsize=(20, 15))

# 评论星级标准差与成功指标的关系
plt.subplot(2, 3, 1)
sns.scatterplot(x='std_review_stars', y='success_score', data=business_features)
plt.title('Std of Review Stars vs Success Score')
sns.regplot(x='std_review_stars', y='success_score', data=business_features, 
           scatter=False, line_kws={"color":"red"})
plt.title('Std of Review Stars vs Success Score')

# 商家年龄与成功指标的关系
plt.subplot(2, 3, 2)
sns.scatterplot(x='business_age_days', y='success_score', data=business_features)
plt.title('Business Age (Days) vs Success Score')
plt.xscale('log')

# 每日评论数与成功指标的关系
plt.subplot(2, 3, 3)
sns.scatterplot(x='reviews_per_day', y='success_score', data=business_features)
plt.title('Reviews Per Day vs Success Score')
plt.xscale('log')

# 城市规模与成功指标的关系
plt.subplot(2, 3, 4)
sns.boxplot(x='is_in_major_city', y='success_score', data=business_features)
plt.title('Major City vs Success Score')

# 各州商家成功指标的分布
plt.subplot(2, 3, 5)
top_states = business_features['state'].value_counts().nlargest(10).index
state_data = business_features[business_features['state'].isin(top_states)]
sns.boxplot(x='state', y='success_score', data=state_data)
plt.title('State vs Success Score')

plt.tight_layout()
plt.show()

In [ ]:
#构建自变量和因变量dataframe
#由于stars和comments已经作为计算因变量的因子，故从特征中将他们去除，防止数据泄露
X = business_features.drop(['business_id', 'name', 'address', 'city', 'postal_code', 
                              'categories', 'success_score', 'attributes','hours','state','category_first_word','latitude','longitude',
                              'weighted_stars', 'first_review_date','category_group', 'last_review_date','total_reviews','stars','review_count'], axis=1)
y = businesses['success_score']

In [ ]:
# 计算相关系数矩阵
correlation_matrix = X.corr()

# 绘制热力图
plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f', cbar=True)
plt.title('Correlation Heatmap of Features', fontsize=16)
plt.show()

In [ ]:
# 划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 定义模型评估函数
def evaluate_model(model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    # 计算评估指标
    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    
    return {
        'model': model.__class__.__name__,
        'R²': r2,
        'RMSE': rmse,
        'MAE': mae
    }

# 创建和评估多个模型
models = [
    LinearRegression(),
    Ridge(alpha=1.0),
    Lasso(alpha=0.1),
    RandomForestRegressor(n_estimators=100, random_state=42),
    GradientBoostingRegressor(n_estimators=100, random_state=42)
]

results = []
for model in models:
    result = evaluate_model(model, X_train, X_test, y_train, y_test)
    results.append(result)

# 显示模型性能比较
results_df = pd.DataFrame(results)
print(results_df)
  
# 选择最佳模型进行特征重要性分析
best_model_idx = results_df['R²'].argmax()
best_model = models[best_model_idx]
print(f"\n最佳模型: {best_model.__class__.__name__}")

if hasattr(best_model, 'feature_importances_'):
    best_model.fit(X_train, y_train)
    # 提取特征重要性
    feature_importance = pd.DataFrame({
        'Feature': X.columns,
        'Importance': best_model.feature_importances_
    })
    feature_importance = feature_importance.sort_values('Importance', ascending=False)
    
    # 可视化特征重要性
    plt.figure(figsize=(12, 8))
    sns.barplot(x='Importance', y='Feature', data=feature_importance.head())
    plt.title(f'Top 20 Feature Importance ({best_model.__class__.__name__})')
    plt.tight_layout()
    plt.show()
    
    print("\n最重要的10个特征:")
    print(feature_importance.head())

# 时间序列分析

In [ ]:
# 提取时间特征
reviews['year'] = reviews['date'].dt.year  # 提取年份
reviews['month'] = reviews['date'].dt.month  # 提取月份
reviews['day'] = reviews['date'].dt.day  # 提取日期
reviews['dayofweek'] = reviews['date'].dt.dayofweek  # 提取星期几
reviews['is_weekend'] = reviews['dayofweek'].apply(lambda x: 1 if x >= 5 else 0)  # 标记是否为周末
reviews['yearmonth'] = reviews['date'].dt.to_period('M')  # 创建年月组合的时间周期

# 按年度分析 - 计算每年的平均评分和评论数量
yearly_ratings = reviews.groupby('year')['stars'].agg(['mean', 'count']).reset_index()
yearly_ratings.columns = ['year', 'avg_rating', 'review_count']

# 按月度分析 - 计算每月的平均评分和评论数量
monthly_ratings = reviews.groupby('yearmonth')['stars'].agg(['mean', 'count']).reset_index()
monthly_ratings.columns = ['yearmonth', 'avg_rating', 'review_count']
monthly_ratings['yearmonth'] = monthly_ratings['yearmonth'].dt.to_timestamp()  # 转换为时间戳以便绘图

# 按星期几分析 - 计算每个星期几的平均评分和平均评论数量
weekday_ratings = reviews.groupby('dayofweek')['stars'].agg(['mean', 'count']).reset_index()
weekday_ratings.columns = ['dayofweek', 'avg_rating', 'review_count']
weekday_names = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
weekday_ratings['weekday_name'] = weekday_ratings['dayofweek'].apply(lambda x: weekday_names[x])

# 按工作日/周末分析 - 对比工作日和周末的评分差异
weekend_ratings = reviews.groupby('is_weekend')['stars'].agg(['mean', 'count']).reset_index()
weekend_ratings.columns = ['is_weekend', 'avg_rating', 'review_count']
weekend_ratings['type'] = weekend_ratings['is_weekend'].apply(lambda x: 'Weekend' if x == 1 else 'Weekday')

plt.figure(figsize=(14, 18))
# 年度评分趋势 - 第一个子图
plt.subplot(4, 1, 1)
sns.lineplot(x='year', y='avg_rating', data=yearly_ratings, marker='o', linewidth=2)
plt.title('Annual Average Rating Trend', fontsize=14)
plt.xlabel('Year')
plt.ylabel('Average Rating')
plt.grid(True)
# 添加平均值参考线
plt.axhline(y=yearly_ratings['avg_rating'].mean(), color='r', linestyle='--', alpha=0.5)
plt.text(yearly_ratings['year'].iloc[0], yearly_ratings['avg_rating'].mean(), 
         f' Average: {yearly_ratings["avg_rating"].mean():.2f}', va='center')

# 年度评论数量趋势 - 第二个子图
plt.subplot(4, 1, 2)
sns.lineplot(x='year', y='review_count', data=yearly_ratings, marker='o', linewidth=2, color='orange')
plt.title('Annual Review Count Trend', fontsize=14)
plt.xlabel('Year')
plt.ylabel('Review Count')
plt.grid(True)

# 月度评分趋势 - 第三个子图
plt.subplot(4, 1, 3)
sns.lineplot(x='yearmonth', y='avg_rating', data=monthly_ratings, marker='.', linewidth=1)
plt.title('Monthly Average Rating Trend', fontsize=14)
plt.xlabel('Year-Month')
plt.ylabel('Average Rating')
plt.grid(True)
plt.xticks(rotation=45)
# 设置x轴显示格式，以年为单位显示刻度
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.gca().xaxis.set_major_locator(mdates.YearLocator())

# 月度评论数量趋势 - 第四个子图
plt.subplot(4, 1, 4)
sns.lineplot(x='yearmonth', y='review_count', data=monthly_ratings, marker='.', linewidth=1, color='orange')
plt.title('Monthly Review Count Trend', fontsize=14)
plt.xlabel('Year-Month')
plt.ylabel('Review Count')
plt.grid(True)
plt.xticks(rotation=45)
# 设置x轴显示格式，以年为单位显示刻度
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.gca().xaxis.set_major_locator(mdates.YearLocator())
plt.tight_layout()  
plt.show()

In [ ]:
plt.figure(figsize=(16, 8))
# 按星期几的平均评分 - 左侧子图
plt.subplot(1, 2, 1)
sns.barplot(x='weekday_name', y='avg_rating', data=weekday_ratings, palette='viridis')
plt.title('Average Rating by Weekday', fontsize=14)
plt.xlabel('Weekday')
plt.ylabel('Average Rating')
# 调整y轴范围以更好地显示评分差异
y_min = weekday_ratings['avg_rating'].min() - 0.1
y_max = weekday_ratings['avg_rating'].max() + 0.1
plt.ylim(y_min, y_max)
# 在柱状图上显示具体数值
for i, v in enumerate(weekday_ratings['avg_rating']):
    plt.text(i, v + 0.01, f'{v:.2f}', ha='center')

# 按星期几的评论数量 - 右侧子图
plt.subplot(1, 2, 2)
sns.barplot(x='weekday_name', y='review_count', data=weekday_ratings, palette='viridis')
plt.title('Review Count by Weekday', fontsize=14)
plt.xlabel('Weekday')
plt.ylabel('Review Count')
# 在柱状图上显示具体数值
for i, v in enumerate(weekday_ratings['review_count']):
    plt.text(i, v + 5, f'{v}', ha='center')

plt.tight_layout()
plt.show()

In [ ]:

plt.figure(figsize=(16, 8))
# 工作日/周末的平均评分 - 左侧子图
plt.subplot(1, 2, 1)
sns.barplot(x='type', y='avg_rating', data=weekend_ratings, palette='Set2')
plt.title('Workday vs Weekend Average Rating Comparison', fontsize=14)
plt.xlabel('')
plt.ylabel('Average Rating')
# 调整y轴范围以更好地显示评分差异
y_min = weekend_ratings['avg_rating'].min() - 0.1
y_max = weekend_ratings['avg_rating'].max() + 0.1
plt.ylim(y_min, y_max)
# 在柱状图上显示具体数值
for i, v in enumerate(weekend_ratings['avg_rating']):
    plt.text(i, v + 0.01, f'{v:.2f}', ha='center')

# 工作日/周末的评论数量 - 右侧子图
weekend_ratings['review_count'].iloc[0] = weekend_ratings['review_count'].iloc[0]/5
weekend_ratings['review_count'].iloc[1] = weekend_ratings['review_count'].iloc[1]/2
plt.subplot(1, 2, 2)
sns.barplot(x='type', y='review_count', data=weekend_ratings, palette='Set2')
plt.title('Workday vs Weekend Review Count Comparison', fontsize=14)
plt.xlabel('')
plt.ylabel('Review Count')
# 在柱状图上显示具体数值
for i, v in enumerate(weekend_ratings['review_count']):
    plt.text(i, v + 5, f'{v}', ha='center')

plt.tight_layout()
plt.show()

In [ ]:

# 按月份分析评分和评论数量
monthly_pattern = reviews.groupby('month')['stars'].agg(['mean', 'count']).reset_index()
monthly_pattern.columns = ['month', 'avg_rating', 'review_count']
plt.figure(figsize=(16, 8))

# 不同月份的平均评分 - 左侧子图
plt.subplot(1, 2, 1)
sns.lineplot(x='month', y='avg_rating', data=monthly_pattern, marker='o', linewidth=2)
plt.title('Average Rating by Month', fontsize=14)
plt.xlabel('Month')
plt.ylabel('Average Rating')
plt.grid(True)
# 设置x轴刻度为月份名称
month_names = ['January', 'February', 'March', 'April', 'May', 'June', 'July', 'August', 'September', 'October', 'November', 'December']
plt.xticks(range(1, 13), month_names, rotation=45)
y_min = monthly_pattern['avg_rating'].min() - 0.05
y_max = monthly_pattern['avg_rating'].max() + 0.05
plt.ylim(y_min, y_max)

# 不同月份的评论数量 - 右侧子图
plt.subplot(1, 2, 2)
sns.lineplot(x='month', y='review_count', data=monthly_pattern, marker='o', linewidth=2, color='orange')
plt.title('Review Count by Month', fontsize=14)
plt.xlabel('Month')
plt.ylabel('Review Count')
plt.grid(True)
plt.xticks(range(1, 13), month_names, rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# 创建月度时间序列
reviews['month_year'] = pd.to_datetime(reviews['date']).dt.to_period('M')
monthly_series = reviews.groupby('month_year')['stars'].mean()
monthly_series.index = monthly_series.index.to_timestamp()

#加法季节性分解
decomposition = seasonal_decompose(monthly_series, model='additive', period=12)
plt.figure(figsize=(14, 12))
# 原始时间序列 - 第一个子图
plt.subplot(4, 1, 1)
plt.plot(decomposition.observed)
plt.title('Original Rating Time Series', fontsize=14)
plt.xlabel('')
plt.ylabel('Rating')
plt.grid(True)
# 趋势成分 - 第二个子图
plt.subplot(4, 1, 2)
plt.plot(decomposition.trend)
plt.title('Trend Component - Reflects Long-Term Changes', fontsize=14)
plt.xlabel('')
plt.ylabel('Trend')
plt.grid(True)

# 季节成分 - 第三个子图
plt.subplot(4, 1, 3)
plt.plot(decomposition.seasonal)
plt.title('Seasonal Component - Reflects Periodic Changes', fontsize=14)
plt.xlabel('')
plt.ylabel('Seasonality')
plt.grid(True)

# 残差成分 - 第四个子图
plt.subplot(4, 1, 4)
plt.plot(decomposition.resid)
plt.title('Residual Component - Reflects Random Fluctuations', fontsize=14)
plt.xlabel('Date')
plt.ylabel('Residual')
plt.grid(True)

plt.tight_layout()
plt.show()
